In [1]:
# Ejecutar solo la primera vez:
#!pip install semopy statsmodels -q

###Importación de librerías a utilizar:

import warnings

import numpy as np
import pandas as pd
import semopy
import statsmodels.api as sm
from scipy import stats
from scipy.stats import zscore
from pathlib import Path

warnings.filterwarnings("ignore")


###Parámetros generales:

RUTA_CSV = Path("C:/Users/Usuario/Downloads/Datos brutos ESS.csv")
CODIGOS_MISSING = [77, 88, 99, 777, 888, 999]

# Variables del SWB y las que se invierten (escala 1-4 → 5 - x)
VARS_SWB = ["happy", "stflife", "wrhpp", "enjlf",
            "fltsd", "fltdpr", "fltlnl", "cldgng", "anweight"]
VARS_A_INVERTIR = ["fltsd", "fltdpr", "fltlnl", "cldgng"]

# Variables del modelo de regresión (trabajadores remunerados)
VIS_CONTINUAS = ["hincfel_inv", "wkhtot", "wkdcorga"]
VIS_MODELO = ["hincfel_inv_c", "wkhtot_c", "wrkctra_limited",
              "wrkctra_nocontract", "wkdcorga_c", "jbspv_r"]

NOMBRES_VIS = {
    "const":              "Constante",
    "hincfel_inv_c":      "Comodidad con los ingresos (hincfel invertida, 1 = muy difícil a 4 = cómodo)",
    "wkhtot_c":           "Horas trabajadas (wkhtot)",
    "wrkctra_limited":    "Contrato limitado vs. indefinido",
    "wrkctra_nocontract": "Sin contrato vs. indefinido",
    "wkdcorga_c":         "Autonomía organizativa (wkdcorga, 0-10)",
    "jbspv_r":            "Supervisa a otros (jbspv, 1 = sí)",
}
# Códigos de no respuesta específicos de cada variable
MISSING_POR_VARIABLE = {
    "happy":    [77, 88, 99],
    "stflife":  [77, 88, 99],
    "wrhpp":    [7, 8, 9],
    "enjlf":    [7, 8, 9],
    "fltsd":    [7, 8, 9],
    "fltdpr":   [7, 8, 9],
    "fltlnl":   [7, 8, 9],
    "cldgng":   [7, 8, 9],
    "anweight": [],
}

###################1. Funciones auxiliares:

def cargar_datos(ruta=RUTA_CSV):
    """Lee el CSV del ESS y limpia los nombres de columnas."""
    df = pd.read_csv(ruta, sep=";", decimal=",")
    df.columns = df.columns.str.strip().str.replace('"', "")
    return df


def a_numerico(df, columnas, missing=None):
    """
    Convierte a numérico. `missing` es un dict {variable: [códigos]}
    para sustituir los códigos de no respuesta por NaN de cada variable.
    """
    df = df.copy()
    for col in columnas:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        if missing and missing.get(col):
            df[col] = df[col].replace(missing[col], np.nan)
    return df


def correlacion_ponderada(x, y, w):
    """Correlación de Pearson ponderada."""
    w = w / w.sum()
    mx, my = np.sum(w * x), np.sum(w * y)
    cov = np.sum(w * (x - mx) * (y - my))
    sx = np.sqrt(np.sum(w * (x - mx) ** 2))
    sy = np.sqrt(np.sum(w * (y - my) ** 2))
    return cov / (sx * sy)


def replicar_por_peso(df, col_peso="anweight", factor=10):
    """Ponderación manual: replica filas según el peso normalizado."""
    peso_norm = df[col_peso] / df[col_peso].mean()
    reps = (peso_norm * factor).round().astype(int).clip(lower=1)
    return df.loc[df.index.repeat(reps)].reset_index(drop=True)


def imprimir_titulo(texto):
    print("\n" + "=" * 60)
    print(texto)
    print("=" * 60)


def tabla_coeficientes(modelo, nombres=None, ocultar="pais_"):
    """Tabla B, SE, t, p (sin las dummies de país)."""
    idx = [v for v in modelo.params.index if not v.startswith(ocultar)]
    nombres_var = [nombres.get(v, v) if nombres else v for v in idx]
    return pd.DataFrame({
        "Variable": nombres_var,
        "B":  modelo.params[idx].values.round(3),
        "SE": modelo.bse[idx].values.round(3),
        "t":  modelo.tvalues[idx].values.round(3),
        "p":  [f"{p:.3f}" for p in modelo.pvalues[idx]],
    })


def imprimir_resumen_modelo(modelo, nombres=None):
    print(tabla_coeficientes(modelo, nombres).to_string(index=False))
    print(f"\nN = {int(modelo.nobs)} | países (clusters) = {modelo.n_paises}")
    print(f"R² ponderado = {modelo.rsquared:.3f} | R² ajustado = {modelo.rsquared_adj:.3f}")


def efectos_pais(df):
    """Dummies de país (efectos fijos)."""
    return pd.get_dummies(df["cntry"], prefix="pais", drop_first=True).astype(float)


def ajustar_wls(df, X):
    """WLS con anweight y errores estándar agrupados por país (p con t, gl = clusters - 1)."""
    res = sm.WLS(df["happy"].astype(float), X.astype(float),
                 weights=df["anweight"].astype(float)).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(df["cntry"])[0]},
        use_t=True)
    res.n_paises = df["cntry"].nunique()
    return res


def test_interacciones(modelo, etiqueta):
    """Test de Wald conjunto del bloque de interacciones, global y por variable."""
    inter = [n for n in modelo.params.index if "_x_" in n]
    t = modelo.wald_test(", ".join(f"{n} = 0" for n in inter), use_f=True, scalar=True)
    print(f"\n{etiqueta} | bloque completo: F({int(t.df_num)}, {int(t.df_denom)}) = "
          f"{float(t.statistic):.2f}, p = {float(t.pvalue):.4f}")
    for vi in VIS_MODELO:
        sub = [n for n in inter if n.startswith(f"{vi}_x_")]
        if sub:
            t = modelo.wald_test(", ".join(f"{n} = 0" for n in sub), use_f=True, scalar=True)
            print(f"   {vi:<20} F({int(t.df_num)}, {int(t.df_denom)}) = "
                  f"{float(t.statistic):.2f}, p = {float(t.pvalue):.4f}")

            
#################2. SEM del SWB: comparación de dos modelos

def corr_ponderada_matriz(X, w):
    """Matriz de correlaciones ponderada."""
    w = np.asarray(w, float)
    C = np.cov(X.values.T, aweights=w)
    d = np.sqrt(np.diag(C))
    return pd.DataFrame(C / np.outer(d, d), index=X.columns, columns=X.columns)


def n_efectivo(w):
    """N efectivo de Kish."""
    w = np.asarray(w, float)
    return w.sum() ** 2 / (w ** 2).sum()


def preparar_datos_swb(df_raw):
    """Limpia códigos de no respuesta, elimina filas incompletas e invierte los ítems negativos."""
    df = a_numerico(df_raw[VARS_SWB], VARS_SWB, missing=MISSING_POR_VARIABLE)
    if "cntry" in df_raw.columns:
        df["cntry"] = df_raw["cntry"].values
    df = df.dropna(subset=VARS_SWB).reset_index(drop=True)
    for v in VARS_A_INVERTIR:                 # escala 1-4 -> 5 - x
        df[f"{v}_inv"] = 5 - df[v]
    print(f"N total tras limpieza: {len(df)}")
    return df

def ejecutar_sem(df, desc, columnas, titulo, ponderado=True):
    """Ajusta un modelo con la misma matriz S para todos los modelos."""
    X = df[columnas]
    if ponderado:
        S = corr_ponderada_matriz(X, df["anweight"])
        n = int(n_efectivo(df["anweight"]))
    else:
        S = X.corr()
        n = len(X)

    model = semopy.Model(desc)
    model.fit(cov=S, n_samples=n)

    imprimir_titulo(f"{titulo}  (N bruto = {len(X)} | N en los índices = {n})")
    resultados = model.inspect(std_est=True).copy()
    resultados["p-value"] = resultados["p-value"].apply(
        lambda x: f"{float(x):.3f}" if x not in ["-", "", None] and pd.notna(x) else "-")
    print(resultados.to_string(index=True))

    print("\n=== ÍNDICES DE AJUSTE ===")
    print(semopy.calc_stats(model).T.drop(index=["AIC", "BIC", "LogLik"], errors="ignore").round(3))
    return model


def comparar_modelos(modelos, titulo, n):
    """Ajuste de varios modelos. AIC/BIC calculados desde χ² (los de semopy no son válidos)."""
    p = len(cols_sem)
    momentos = p * (p + 1) // 2
    filas = {}
    for nombre, m in modelos.items():
        s = semopy.calc_stats(m).iloc[0]
        chi2, gl = float(s["chi2"]), float(s["DoF"])
        q = momentos - gl
        filas[nombre] = {"χ²": chi2, "gl": gl,
                         "CFI": s.get("CFI"), "TLI": s.get("TLI"),
                         "RMSEA": s.get("RMSEA"),
                         "AIC": chi2 + 2 * q,
                         "BIC": chi2 + q * np.log(n)}
    tabla = pd.DataFrame(filas).T.astype(float)
    imprimir_titulo(titulo)
    print(tabla.round(3).to_string())

    nombres = list(modelos)
    if len(nombres) == 2:
        a, b = tabla.loc[nombres[0]], tabla.loc[nombres[1]]
        dchi, ddf = b["χ²"] - a["χ²"], b["gl"] - a["gl"]
        if ddf > 0:
            print(f"\nContraste {nombres[1]} vs {nombres[0]}: "
                  f"Δχ² = {dchi:.2f}, Δgl = {ddf:.0f}, p = {stats.chi2.sf(dchi, ddf):.4g}")
    print(f"\nMenor AIC: {tabla['AIC'].idxmin()} | Menor BIC: {tabla['BIC'].idxmin()}")
    return tabla


def diagnostico_peso(df):
    imprimir_titulo("DIAGNÓSTICO DEL PESO: happy vs stflife")
    r_sin = df["happy"].corr(df["stflife"])
    r_pon = correlacion_ponderada(df["happy"], df["stflife"], df["anweight"])
    print(f"r sin ponderar = {r_sin:.3f} | r ponderada = {r_pon:.3f}")
    print(f"N efectivo (Kish) = {n_efectivo(df['anweight']):.0f} de {len(df)}")
    print("Peso mín/mediana/máx =",
          round(df["anweight"].min(), 3),
          round(df["anweight"].median(), 3),
          round(df["anweight"].max(), 3))
    if "cntry" in df.columns:
        print("\nPaíses incluidos en el SEM:")
        print(df["cntry"].value_counts())
        res = []
        for c, g in df.groupby("cntry"):
            if len(g) > 100:
                res.append({"cntry": c, "n": len(g),
                            "peso_medio": g["anweight"].mean(),
                            "r_sin": g["happy"].corr(g["stflife"]),
                            "r_pond": correlacion_ponderada(
                                g["happy"], g["stflife"], g["anweight"])})
        print("\n", pd.DataFrame(res).sort_values("peso_medio", ascending=False)
                      .round(3).to_string(index=False))


###################2.1 Definición de los dos modelos

desc_A = """
AP =~ wrhpp + enjlf
AN =~ fltsd_inv + fltdpr_inv + fltlnl_inv
SV =~ stflife + cldgng_inv
SWB =~ AP + AN + SV
happy ~ SWB
"""

desc_B = """
AP =~ wrhpp + enjlf + happy
AN =~ fltsd_inv + fltdpr_inv + fltlnl_inv
SV =~ cldgng_inv
SWB =~ AP + AN + SV
stflife ~ SWB
"""

cols_sem = ["wrhpp", "enjlf", "fltsd_inv", "fltdpr_inv", "fltlnl_inv",
            "cldgng_inv", "stflife", "happy"]


df_raw = cargar_datos()
df_swb = preparar_datos_swb(df_raw)

#####################2.2 Ajuste ponderado (análisis principal)

mA = ejecutar_sem(df_swb, desc_A, cols_sem, "MODELO A: happy carga en SWB", ponderado=True)
mB = ejecutar_sem(df_swb, desc_B, cols_sem, "MODELO B: stflife carga en SWB", ponderado=True)
n_pond = int(n_efectivo(df_swb["anweight"]))
tabla_pond = comparar_modelos({"A (happy en SWB)": mA, "B (stflife en SWB)": mB},
                              "COMPARACIÓN PONDERADA", n_pond)

###################2.3 Ajuste sin ponderar (sensibilidad)

mA_s = ejecutar_sem(df_swb, desc_A, cols_sem, "MODELO A sin ponderar", ponderado=False)
mB_s = ejecutar_sem(df_swb, desc_B, cols_sem, "MODELO B sin ponderar", ponderado=False)
tabla_sin = comparar_modelos({"A (happy en SWB)": mA_s, "B (stflife en SWB)": mB_s},
                             "COMPARACIÓN SIN PONDERAR", len(df_swb))

#######################2.4 Diagnóstico del peso

diagnostico_peso(df_swb)

###################2.5 Criterio externo: AFC sin la candidata + correlación del factor con ella

# --- Funciones auxiliares (déjalas como estaban) ---
def estandarizar_ponderado(X, w):
    """Estandariza con media y desviación típica ponderadas (coherente con la matriz S del AFC)."""
    w = np.asarray(w, float) / np.sum(w)
    m = X.mul(w, axis=0).sum()
    sd = np.sqrt(((X - m) ** 2).mul(w, axis=0).sum())
    return (X - m) / sd


def ic_fisher(r, n, nivel=0.95):
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    k = stats.norm.ppf(0.5 + nivel / 2)
    return np.tanh(z - k * se), np.tanh(z + k * se)


def cov_factor_indicadores(modelo, columnas, factor="SWB"):
    """Correlación de `factor` con cada indicador = producto de cargas estandarizadas."""
    ins = modelo.inspect(std_est=True)
    ins = ins[ins["op"] == "~"]
    b = {(r["lval"], r["rval"]): float(r["Est. Std"]) for _, r in ins.iterrows()}

    def efecto(nodo):
        if nodo == factor:
            return 1.0
        return sum(coef * efecto(padre) for (hijo, padre), coef in b.items() if hijo == nodo)

    return np.array([efecto(c) for c in columnas])


def criterio_externo(df, modelo, columnas, candidata, titulo, ponderado=True, factor="SWB"):
    w = df["anweight"].values if ponderado else np.ones(len(df))
    n = int(n_efectivo(w))
    Z = estandarizar_ponderado(df[columnas], w)

    S = corr_ponderada_matriz(df[columnas], w).values
    c = cov_factor_indicadores(modelo, columnas, factor)
    pesos = np.linalg.solve(S, c)            # método de regresión (Thurstone)
    f = Z.values @ pesos

    r = correlacion_ponderada(f, df[candidata].values.astype(float), w)
    lo, hi = ic_fisher(r, n)
    imprimir_titulo(f"CRITERIO EXTERNO – {titulo}  (N efectivo = {n})")
    print(f"r(factor SWB, {candidata}) = {r:.3f}  IC95% [{lo:.3f}, {hi:.3f}]")
    return {"candidata": candidata, "r": r, "IC_inf": lo, "IC_sup": hi, "n_ef": n}

# --- AFC 1: sin happy (stflife dentro del factor) ---
desc_sin_happy = """
AP =~ wrhpp + enjlf
AN =~ fltsd_inv + fltdpr_inv + fltlnl_inv
SV =~ stflife + cldgng_inv
SWB =~ AP + AN + SV
"""
cols_sin_happy = ["wrhpp", "enjlf", "fltsd_inv", "fltdpr_inv", "fltlnl_inv",
                  "cldgng_inv", "stflife"]

# --- AFC 2: sin stflife (happy dentro del factor) ---
desc_sin_stflife = """
AP =~ wrhpp + enjlf + happy
AN =~ fltsd_inv + fltdpr_inv + fltlnl_inv
SWB =~ AP + AN + cldgng_inv
"""
cols_sin_stflife = ["wrhpp", "enjlf", "fltsd_inv", "fltdpr_inv", "fltlnl_inv",
                    "cldgng_inv", "happy"]

resultados_ext = []
for ponderado, etiq in [(True, "ponderado"), (False, "sin ponderar")]:
    m1 = ejecutar_sem(df_swb, desc_sin_happy, cols_sin_happy,
                      f"AFC sin happy ({etiq})", ponderado=ponderado)
    r1 = criterio_externo(df_swb, m1, cols_sin_happy, "happy",
                          f"factor sin happy, {etiq}", ponderado=ponderado)

    m2 = ejecutar_sem(df_swb, desc_sin_stflife, cols_sin_stflife,
                      f"AFC sin stflife ({etiq})", ponderado=ponderado)
    r2 = criterio_externo(df_swb, m2, cols_sin_stflife, "stflife",
                          f"factor sin stflife, {etiq}", ponderado=ponderado)
    resultados_ext += [{"Análisis": etiq, "AFC": "sin happy (stflife dentro)", **r1},
                       {"Análisis": etiq, "AFC": "sin stflife (happy dentro)", **r2}]

imprimir_titulo("RESUMEN CRITERIO EXTERNO")
print(pd.DataFrame(resultados_ext).round(3).to_string(index=False))
print("\nNota: cada AFC tiene indicadores distintos, así que χ², AIC y BIC NO son comparables "
      "y no se usan. Además, el factor de cada AFC incluye a la otra variable "
      "(stflife en el primero, happy en el segundo), por lo que las dos correlaciones "
      "tampoco están en igualdad de condiciones. Se interpretan de forma descriptiva "
      "(con sus IC), sin contraste formal entre ellas.")
##########################3. Modelo 2: Regresión lineal ponderada (WLS)

def preparar_trabajo_pagado(df_raw, extra_cols=()):
    """
    Filtra trabajadores remunerados, elimina códigos missing explícitos,
    recodifica y centra las VIs. `extra_cols` permite añadir moderadores.
    """
    cols = ["happy", "mnactic", "hincfel", "wkhtot", "wrkctra",
            "wkdcorga", "jbspv", "anweight", *extra_cols]
    df = a_numerico(df_raw[cols], cols)
    df["cntry"] = df_raw["cntry"]          # texto: se añade después de convertir a numérico

    # Filtros: muestra y códigos missing explícitos
    df = df[df["mnactic"] == 1]
    df = df[df["happy"].le(10)]
    df = df[df["hincfel"].isin([1, 2, 3, 4])]
    df = df[df["wkhtot"].between(1, 80)]    
    df = df[df["wrkctra"].isin([1, 2, 3])]
    df = df[df["wkdcorga"].le(10)]
    df = df[df["jbspv"].isin([1, 2])]

# Moderadores: quitar no respuesta (se mantienen las 3 categorías de género)
    if "agegroup" in df:
        df = df[df["agegroup"].between(1, 7)]
    if "nobingnd" in df:
        df = df[df["nobingnd"].isin([1, 2, 3])]

    df = df.dropna().reset_index(drop=True)

    df["hincfel_inv"] = 5 - df["hincfel"] 

    # Recodificaciones
    df["jbspv_r"] = (df["jbspv"] == 1).astype(int)                 # 1 = supervisa
    df["wrkctra_limited"] = (df["wrkctra"] == 2).astype(int)       # ref = indefinido
    df["wrkctra_nocontract"] = (df["wrkctra"] == 3).astype(int)

    # Centrado de continuas/ordinales
    for var in VIS_CONTINUAS:
        df[f"{var}_c"] = df[var] - df[var].mean()
    return df

df_m2 = preparar_trabajo_pagado(df_raw)
print(f"N grupo 1 (trabajo pagado) tras limpieza: {len(df_m2)}")
print("\nEstadísticos descriptivos (continuas):")
print(df_m2[["happy", "hincfel_inv", "wkhtot", "wkdcorga"]].describe().round(3))
print("\nTipo de contrato (proporciones):")
print(df_m2["wrkctra"].value_counts(normalize=True).sort_index().round(3))
print("\nSupervisa a otros (proporción):")
print(df_m2["jbspv_r"].value_counts(normalize=True).sort_index().round(3))

X_m2 = sm.add_constant(df_m2[VIS_MODELO])
modelo_wls = ajustar_wls(df_m2, X_m2)

imprimir_titulo("MODELO 2 – REGRESIÓN LINEAL PONDERADA (WLS)\n"
                "Grupo 1 mnactic: trabajadores remunerados")
imprimir_resumen_modelo(modelo_wls, NOMBRES_VIS)
print("\nNota: referencia wrkctra = contrato indefinido (1)")
print("Nota: jbspv recodificada: 1 = supervisa, 0 = no supervisa")

###como análisis de robustez, puedes añadir un logit ordinal para el Modelo 2 (que es lo sgiuiente)
from statsmodels.miscmodels.ordinal_model import OrderedModel

X_ord = df_m2[VIS_MODELO].astype(float)
ord_res = OrderedModel(df_m2["happy"].astype(int), X_ord, distr="logit").fit(
    method="bfgs", disp=False,
    cov_type="cluster", cov_kwds={"groups": pd.factorize(df_m2["cntry"])[0]})
print(ord_res.summary().tables[1])

###########################4. Modelo 3: Moderación por género y edad

def ajustar_moderacion(df, moderadores):
    """
    moderadores: dict {prefijo_dummy: columna}, p. ej. {'genero': 'nobingnd'}.
    Devuelve el modelo WLS con efectos principales, dummies e interacciones.
    """
    dummies = {
        pref: pd.get_dummies(df[col], prefix=pref, drop_first=True).astype(float)
        for pref, col in moderadores.items()
    }

    interacciones = []
    for dum in dummies.values():
        inter = pd.DataFrame({
            f"{vi}_x_{c}": df[vi].values * dum[c].values
            for vi in VIS_MODELO for c in dum.columns
        })
        interacciones.append(inter)

    X = pd.concat([df[VIS_MODELO], *dummies.values(), *interacciones], axis=1).astype(float)
    X = sm.add_constant(X)
    return ajustar_wls(df, X)

df_m3 = preparar_trabajo_pagado(df_raw, extra_cols=["nobingnd", "agegroup"])
print(f"N grupo 1 (trabajo pagado) tras limpieza: {len(df_m3)}")

############################4.1 Modelo 3A: género

modelo_genero = ajustar_moderacion(df_m3, {"genero": "nobingnd"})
imprimir_titulo("MODELO 3A – MODERACIÓN POR GÉNERO (nobingnd)")
imprimir_resumen_modelo(modelo_genero)
test_interacciones(modelo_genero, "3A género")

###############################4.2 Modelo 3B: edad

modelo_edad = ajustar_moderacion(df_m3, {"edad": "agegroup"})
imprimir_titulo("MODELO 3B – MODERACIÓN POR EDAD (agegroup)")
imprimir_resumen_modelo(modelo_edad)
test_interacciones(modelo_edad, "3B edad")

####################################4.3 Modelo 3C: género y edad conjuntos

modelo_conjunto = ajustar_moderacion(df_m3, {"genero": "nobingnd", "edad": "agegroup"})
imprimir_titulo("MODELO 3C – MODERACIÓN CONJUNTA GÉNERO Y EDAD")
imprimir_resumen_modelo(modelo_conjunto)
test_interacciones(modelo_conjunto, "3C conjunto")

###############################4.4 Modelo 3D: triple interacción (condición laboral × género × edad)

def ajustar_moderacion_triple(df, col_a="nobingnd", col_b="edad3",
                              pref_a="genero", pref_b="edad"):
    """
    WLS con: efectos principales + dummies de género y edad + género×edad
    + VI×género + VI×edad + VI×género×edad.
    El df debe tener índice 0..n-1 (reset_index) para que concat alinee bien.
    """
    da = pd.get_dummies(df[col_a], prefix=pref_a, drop_first=True).astype(float)
    db = pd.get_dummies(df[col_b], prefix=pref_b, drop_first=True).astype(float)

    ab  = pd.DataFrame({f"{a}_x_{b}": da[a].values * db[b].values
                        for a in da.columns for b in db.columns})
    va  = pd.DataFrame({f"{vi}_x_{a}": df[vi].values * da[a].values
                        for vi in VIS_MODELO for a in da.columns})
    vb  = pd.DataFrame({f"{vi}_x_{b}": df[vi].values * db[b].values
                        for vi in VIS_MODELO for b in db.columns})
    vab = pd.DataFrame({f"{vi}_x_{a}_x_{b}": df[vi].values * da[a].values * db[b].values
                        for vi in VIS_MODELO for a in da.columns for b in db.columns})

    X = pd.concat([df[VIS_MODELO], da, db, ab, va, vb, vab], axis=1).astype(float)
    X = sm.add_constant(X)
    return ajustar_wls(df, X)


def _wald_bloque(modelo, nombres_bloque):
    """Wald F conjunto (β = 0) para una lista de parámetros, usando matriz R."""
    cols = list(modelo.params.index)
    R = np.zeros((len(nombres_bloque), len(cols)))
    for i, n in enumerate(nombres_bloque):
        R[i, cols.index(n)] = 1
    return modelo.wald_test(R, use_f=True, scalar=True)


def test_triple(modelo):
    """Contrastes de Wald: género×edad, triple global y triple por variable."""
    nombres = list(modelo.params.index)
    triples = [n for n in nombres if n.count("_x_") == 2]
    gen_edad = [n for n in nombres if n.count("_x_") == 1
                and n.startswith("genero_") and "_x_edad_" in n]
    g = modelo.n_paises - 1

    def imprimir(etiqueta, bloque):
        t = _wald_bloque(modelo, bloque)
        aviso = "  ⚠ q > nº clusters - 1: contraste poco fiable" if len(bloque) > g else ""
        print(f"{etiqueta:<42} F({int(t.df_num)}, {int(t.df_denom)}) = "
              f"{float(t.statistic):.2f}, p = {float(t.pvalue):.4f}{aviso}")

    imprimir("Género × edad (sobre la felicidad)", gen_edad)
    imprimir("Triple (todas las VIs)", triples)
    imprimir("Género×edad + triple (3D vs 3C)", gen_edad + triples)
    print("\nTriple interacción por variable:")
    for vi in VIS_MODELO:
        sub = [n for n in triples if n.startswith(f"{vi}_x_")]
        if sub:
            imprimir(f"   {vi}", sub)


def pendientes_por_celda(modelo, df, col_a="nobingnd", col_b="edad3",
                         pref_a="genero", pref_b="edad"):
    """Efecto (pendiente) de cada VI en cada combinación género × edad."""
    cols = list(modelo.params.index)
    niv_a, niv_b = sorted(df[col_a].unique()), sorted(df[col_b].unique())
    dum_a = dict(zip(niv_a[1:], pd.get_dummies(df[col_a], prefix=pref_a, drop_first=True).columns))
    dum_b = dict(zip(niv_b[1:], pd.get_dummies(df[col_b], prefix=pref_b, drop_first=True).columns))

    filas = []
    for vi in VIS_MODELO:
        for ga in niv_a:
            for eb in niv_b:
                terminos = [vi]
                if ga in dum_a: terminos.append(f"{vi}_x_{dum_a[ga]}")
                if eb in dum_b: terminos.append(f"{vi}_x_{dum_b[eb]}")
                if ga in dum_a and eb in dum_b:
                    terminos.append(f"{vi}_x_{dum_a[ga]}_x_{dum_b[eb]}")
                R = np.zeros(len(cols))
                for t in terminos:
                    R[cols.index(t)] = 1
                r = modelo.t_test(R)
                filas.append({"Variable": vi, "Género": ga, "Edad": eb,
                              "B": float(np.squeeze(r.effect)),
                              "SE": float(np.squeeze(r.sd)),
                              "p": float(np.squeeze(r.pvalue))})
    return pd.DataFrame(filas).round(3)


# Muestra del modelo 3D (género, edad y condic laborales): solo géneros 1 y 2 (la categoría 3 deja celdas casi vacías con 7 grupos de edad)
df_m4 = df_m3[df_m3["nobingnd"].isin([1, 2])].reset_index(drop=True)   
print(f"N modelo 3D: {len(df_m4)}")
print(pd.crosstab(df_m4["nobingnd"], df_m4["agegroup"]))              

modelo_triple = ajustar_moderacion_triple(df_m4, col_a="nobingnd", col_b="agegroup")
imprimir_titulo("MODELO 3D – TRIPLE INTERACCIÓN (género × edad × condiciones laborales)")
imprimir_resumen_modelo(modelo_triple)
test_triple(modelo_triple)

imprimir_titulo("PENDIENTES POR COMBINACIÓN GÉNERO × EDAD")
print(pendientes_por_celda(modelo_triple, df_m4, col_a="nobingnd", col_b="agegroup").to_string(index=False))

###########################################5. Comparación de modelos

modelos = {"3A (género)": modelo_genero,
           "3B (edad)": modelo_edad,
           "3C (conjunto)": modelo_conjunto}
comparacion = pd.DataFrame({
    "Modelo":      list(modelos),
    "N":           [int(m.nobs) for m in modelos.values()],
    "R²":          [m.rsquared for m in modelos.values()],
    "R² ajustado": [m.rsquared_adj for m in modelos.values()],
    "AIC":         [m.aic for m in modelos.values()],
    "BIC":         [m.bic for m in modelos.values()],
})
imprimir_titulo("COMPARACIÓN DE MODELOS 3A-3C (misma muestra)")
print(comparacion.round(3).to_string(index=False))
print(f"\nMenor AIC: {comparacion.loc[comparacion['AIC'].idxmin(), 'Modelo']} | "
      f"Menor BIC: {comparacion.loc[comparacion['BIC'].idxmin(), 'Modelo']}")

imprimir_titulo("COMPARACIÓN DE MODELOS")
print(comparacion.round(3).to_string(index=False))
print("\nNotas:")
print("- wrkctra: ref = contrato indefinido (1); limited = contrato temporal; nocontract = sin contrato")
print("- jbspv: 1 = supervisa, 0 = no supervisa")
print("- hincfel invertida: 1 = muy difícil, 4 = vive cómodamente (B positivo = más comodidad, más felicidad)")
print("- wkdcorga: 0 = ninguna influencia, 10 = control total")
print("- Errores estándar agrupados por país (28 clusters); p-values con t(27)")
print("- 3D: solo géneros 1 y 2, edad con los 7 grupos de agegroup; su N difiere de 3A-3C, así que el R² no es estrictamente comparable")

N total tras limpieza: 34701

MODELO A: happy carga en SWB  (N bruto = 34701 | N en los índices = 22608)
          lval  op        rval      Estimate      Est. Std  Std. Err     z-value p-value
0           AP   ~         SWB  1.000000e+00  7.419976e-01         -           -       -
1           AN   ~         SWB  9.000025e-01  7.240393e-01  0.012908   69.723806   0.000
2           SV   ~         SWB  1.256310e+00  1.000000e+00  0.014666   85.658615   0.000
3        wrhpp   ~          AP  1.000000e+00  8.250185e-01         -           -       -
4        enjlf   ~          AP  9.630105e-01  7.946098e-01  0.009768   98.592498   0.000
5    fltsd_inv   ~          AN  1.000000e+00  7.609780e-01         -           -       -
6   fltdpr_inv   ~          AN  9.362814e-01  7.124915e-01  0.010448     89.6168   0.000
7   fltlnl_inv   ~          AN  8.582825e-01  6.531580e-01  0.010196   84.180479   0.000
8      stflife   ~          SV  1.000000e+00  7.692440e-01         -           -       -
9   c

          lval  op        rval  Estimate  Est. Std  Std. Err     z-value p-value
0           AP   ~         SWB  1.000000  0.930702         -           -       -
1           AN   ~         SWB  0.877838  0.804638   0.01177   74.580966   0.000
2           SV   ~         SWB  0.704415  0.779888  0.011311   62.276239   0.000
3        wrhpp   ~          AP  1.000000  0.747726         -           -       -
4        enjlf   ~          AP  0.974232  0.728570   0.00981   99.312549   0.000
5        happy   ~          AP  0.992602  0.742369  0.009835   100.92048   0.000
6    fltsd_inv   ~          AN  1.000000  0.759388         -           -       -
7   fltdpr_inv   ~          AN  0.943546  0.716526  0.010218   92.343828   0.000
8   fltlnl_inv   ~          AN  0.856452  0.650326  0.010015   85.518065   0.000
9   cldgng_inv   ~          SV  1.000000  0.628657         -           -       -
10     stflife   ~         SWB  0.991555  0.690132  0.012271   80.804636   0.000
11          AN  ~~          